In [1]:
import ee
import geemap

# 1. Define your specific Google Cloud Project ID here
PROJECT_ID = 'xxxx' # Paste your actual Project ID inside the quotes

# 2. Authenticate and Initialize with the Project ID
try:
    # Try to initialize with existing credentials
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine Initialized Successfully!")
except Exception as e:
    # If credentials fail or don't exist, trigger the auth flow
    print("Authentication required. Please log in.")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine Initialized Successfully!")

# 1. Define the Area of Interest (AOI) - Shifted east to actual agricultural plots
pune_farms = ee.Geometry.Point([74.255028, 18.544278]).buffer(5000) # 5km radius

Earth Engine Initialized Successfully!


In [2]:
import ee
import geemap

# Initialize (Assuming you've already authenticated in your session)
PROJECT_ID = 'xxxx'
try:
    ee.Initialize(project=PROJECT_ID)
except:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# The AOI
pune_farms = ee.Geometry.Point([74.255028, 18.544278]).buffer(5000)

print("Hunting for the clearest single day of satellite data...")

# ==========================================
# 1. THE FIX: Single-Scene Extraction (NO COMPOSITES)
# ==========================================
# Get the SINGLE clearest Landsat 9 image in April
l9_image = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2") \
    .filterBounds(pune_farms) \
    .filterDate('2023-04-01', '2023-04-30') \
    .sort('CLOUD_COVER') \
    .first() \
    .clip(pune_farms)

# Convert to Celsius
thermal_band = l9_image.select('ST_B10')
temp_celsius = thermal_band.multiply(0.00341802).add(149.0).subtract(273.15).rename('Temperature_C')

# Get the SINGLE clearest Sentinel-2 optical scene
s2_image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(pune_farms) \
    .filterDate('2023-04-01', '2023-04-30') \
    .sort('CLOUDY_PIXEL_PERCENTAGE') \
    .first() \
    .clip(pune_farms)

s2_ndvi = s2_image.normalizedDifference(['B8', 'B4']).rename('NDVI')

# ==========================================
# 2. MASKING & SAFE FUSION
# ==========================================
# Because we used .first(), the Landsat image retains its true 30m projection. 
# We can now safely resample it to 10m without the 111km bug!
esa_lc = ee.ImageCollection("ESA/WorldCover/v200").first().clip(pune_farms)
cropland_mask = esa_lc.select('Map').eq(40)
final_mask = s2_ndvi.gt(0.3).And(cropland_mask)

crop_temp = temp_celsius.resample('bilinear').reproject(
    crs=s2_image.select('B8').projection(),
    scale=10
).updateMask(final_mask)

# ==========================================
# 3. Z-SCORE STATISTICS
# ==========================================
stats = crop_temp.reduceRegion(
    reducer=ee.Reducer.mean().combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True),
    geometry=pune_farms,
    scale=10, # Force stat calculation at the 10m field level
    maxPixels=1e9
)

mean_temp = ee.Number(stats.get('Temperature_C_mean'))
std_temp = ee.Number(stats.get('Temperature_C_stdDev'))
crop_z_score = crop_temp.subtract(mean_temp).divide(std_temp)

# ==========================================
# 4. VISUALIZATION
# ==========================================
Map = geemap.Map(height="600px")
Map.centerObject(pune_farms, 13)

# Background Optical
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map.addLayer(s2_image, vis_rgb, "Sentinel-2 Base")

# Z-Score Overlay
# -2 (Deep Blue) = Severely over-watered
# 0 (White) = Average crop temp
# +2 (Deep Red) = Severe water stress
vis_z = {'min': -2, 'max': 2, 'palette': ['blue', 'cyan', 'white', 'yellow', 'red']}
Map.addLayer(crop_z_score, vis_z, "Relative Irrigation Stress (Z-Score)")

print("\n--- NEW SCENE STATISTICS ---")
print(f"Mean Temp: {mean_temp.getInfo():.2f}°C")
print(f"Standard Dev: {std_temp.getInfo():.2f}°C")

Map

Hunting for the clearest single day of satellite data...

--- NEW SCENE STATISTICS ---
Mean Temp: 43.41°C
Standard Dev: 2.70°C


Map(center=[18.54428312944013, 74.2550281500351], controls=(WidgetControl(options=['position', 'transparent_bg…

In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import numpy as np

print("Step 1: Harvesting multidimensional pixel data from Earth Engine...")

# ==========================================
# 1. DATA PREPARATION: The Feature Stack
# ==========================================
# We stack our optical (NDVI) and thermal (Z-Score) data into a single image
ml_image = ee.Image.cat([
    s2_ndvi.rename('NDVI'),
    crop_z_score.rename('Thermal_Z_Score'),
    temp_celsius.rename('Absolute_Temp')
])

# Randomly sample 5,000 pure crop pixels from our 5km radius
# We extract the exact coordinates (geometries=True) so we know exactly where the failures are.
sample_data = ml_image.sample(
    region=pune_farms,
    scale=10,
    numPixels=5000,
    seed=42,
    geometries=True 
)

# ==========================================
# THE FIX: Native Earth Engine to Pandas Extraction
# ==========================================
print("Pulling raw data from Google servers (Bypassing geemap bug)...")

# 1. Pull the raw JSON dictionary directly from Earth Engine
raw_data = sample_data.getInfo()

# 2. Loop through the JSON and extract the agronomic data + GPS coordinates
parsed_features = []
for feature in raw_data['features']:
    row = feature['properties']
    # Because we set geometries=True, we can extract the exact Lat/Lon of the failure
    row['longitude'] = feature['geometry']['coordinates'][0]
    row['latitude'] = feature['geometry']['coordinates'][1]
    parsed_features.append(row)

# 3. Create the DataFrame natively using Pandas
df = pd.DataFrame(parsed_features)
df = df.dropna() # Clean out any null values

print(f"Successfully downloaded {len(df)} sub-hectare crop samples to local memory.")

print(f"Successfully downloaded {len(df)} sub-hectare crop samples to local memory.")

# ==========================================
# 2. UNSUPERVISED MACHINE LEARNING (Isolation Forest)
# ==========================================
print("\nStep 2: Training the AI to find irrigation failures...")

# The model will look at the relationship between Greenness (NDVI) and Relative Heat (Z-Score)
X = df[['NDVI', 'Thermal_Z_Score']]

# Initialize the Isolation Forest. 
# 'contamination=0.05' tells the AI: "Assume about 5% of these pixels represent broken irrigation."
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# Train the model and predict the anomalies in one line
# Returns 1 for Normal, -1 for Anomaly
df['ML_Prediction'] = iso_forest.fit_predict(X)

# ==========================================
# 3. ANALYSIS & RESULTS
# ==========================================
# Filter the DataFrame to show ONLY the catastrophic failures (Anomalies)
anomalies = df[df['ML_Prediction'] == -1]
normal_crops = df[df['ML_Prediction'] == 1]

print(f"\n--- MACHINE LEARNING DIAGNOSTIC REPORT ---")
print(f"Total Farms Analyzed: {len(df)}")
print(f"Healthy/Normal Pixels: {len(normal_crops)}")
print(f"🚨 CRITICAL ANOMALIES DETECTED: {len(anomalies)} 🚨")

print("\nSample of the worst performing field locations (Require immediate agronomic intervention):")
# Sort by highest thermal Z-score to find the absolute hottest, most stressed anomalies
worst_offenders = anomalies.sort_values(by='Thermal_Z_Score', ascending=False).head(5)
print(worst_offenders[['NDVI', 'Thermal_Z_Score', 'Absolute_Temp']])

Step 1: Harvesting multidimensional pixel data from Earth Engine...
Pulling raw data from Google servers (Bypassing geemap bug)...
Successfully downloaded 2548 sub-hectare crop samples to local memory.
Successfully downloaded 2548 sub-hectare crop samples to local memory.

Step 2: Training the AI to find irrigation failures...

--- MACHINE LEARNING DIAGNOSTIC REPORT ---
Total Farms Analyzed: 2548
Healthy/Normal Pixels: 2420
🚨 CRITICAL ANOMALIES DETECTED: 128 🚨

Sample of the worst performing field locations (Require immediate agronomic intervention):
          NDVI  Thermal_Z_Score  Absolute_Temp
1765  0.394694         3.678236      53.460573
1775  0.345650         3.136585      52.100201
151   0.467244         2.953542      51.279877
384   0.303734         2.937569      51.652441
2335  0.569231         2.894986      50.466388


In [4]:
# ==========================================
# Visualizing ML Anomalies
# ==========================================
print("Preparing the tactical visualization map...")

Map_Tactical = geemap.Map(height="700px")
Map_Tactical.centerObject(pune_farms, 13)

# 1. Base Layer: Standard Sentinel-2 Optical
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map_Tactical.addLayer(s2_image, vis_rgb, "Sentinel-2 Base Layer")

# 2. Extract the ML Anomalies from our DataFrame
anomalies_only = df[df['ML_Prediction'] == -1].copy()

# Clean up the data presentation by rounding the decimals 
anomalies_only['NDVI'] = anomalies_only['NDVI'].round(3)
anomalies_only['Thermal_Z_Score'] = anomalies_only['Thermal_Z_Score'].round(2)
anomalies_only['Absolute_Temp'] = anomalies_only['Absolute_Temp'].round(1)

# 3. Add the markers AND the popups
# We pass the exact, valid column names to the popup parameter
Map_Tactical.add_circle_markers_from_xy(
    anomalies_only,
    x='longitude', 
    y='latitude',
    radius=8,
    color='red',
    fill_color='white',
    weight=2,
    opacity=0.9,
    fill_opacity=0.5,
    popup=['NDVI', 'Thermal_Z_Score', 'Absolute_Temp'], 
    layer_name='🚨 CRITICAL FAILURES 🚨'
)

print(f"Mapped {len(anomalies_only)} critical failures. Click the red dots to see the data!")
Map_Tactical

Preparing the tactical visualization map...
Mapped 128 critical failures. Click the red dots to see the data!


Map(center=[18.54428312944013, 74.2550281500351], controls=(WidgetControl(options=['position', 'transparent_bg…